# Bank NPL Rates by Asset Size — Solution

**Goal:** Investigate how a bank’s size (total assets) relates to its non-performing loan (NPL) rate across 158 banks. Full working solutions, alternate implementations, extra practice, and a parameterised simulation.

**Data:** `data/bank_data.csv`

**Audience notes:** Same as the skeleton — adapt depth and language for Credit Risk analysts, subject-matter experts (ALM / supervisors), and time-pressed executives / Board members.

---

## Flowchart of the Desired Outcome

![Bank NPL Analysis Flowchart](bank_npl_flowchart.png)

## Step 0 — Setup & Load Packages

In [ ]:
# load packages
library(ggplot2)
library(readr)
library(dplyr)

## Step 1 — Access the Data

Inspect the first rows and column names.

In [ ]:
# import and inspect data
data <- read_csv("data/bank_data.csv")
head(data)
names(data)
glimpse(data)   # or str(data)

## Step 2 — Isolate the NPL Rate Column as a Vector

In [ ]:
# npl rate vector (dplyr::pull returns a plain vector)
npl_rate <- data %>%
  pull(npl_rate)

# quick check
head(npl_rate)
length(npl_rate)
class(npl_rate)

## Step 3 — Find the Quartiles

In [ ]:
# npl rate quartiles (0%, 25%, 50%, 75%, 100%)
npl_quartiles <- quantile(npl_rate)
npl_quartiles

# also useful:
# quantile(npl_rate, probs = c(0.25, 0.5, 0.75))

## Step 4 — Plot the Histogram of NPL Rates

In [ ]:
# plot histogram of npl rates
hist(npl_rate,
     main = "Distribution of NPL Rates (158 banks)",
     xlab = "NPL Rate (%)",
     col  = "steelblue",
     border = "white")
abline(v = npl_quartiles[2:4], col = c("orange", "red", "darkgreen"), lwd = 2, lty = 2)
legend("topright", legend = c("Q1", "Median", "Q3"),
       col = c("orange", "red", "darkgreen"), lty = 2, lwd = 2, bty = "n")

## Step 5 — Interpret an NPL Rate of 9 %

From the quartiles printed above, locate where 9 % falls.

- If 9 is between the 25 % and 50 % quantiles → second quarter.
- If between the 50 % and 75 % quantiles → third quarter.
- etc.

(In the synthetic data used here the median is typically near 9 %, so 9 % often sits very close to the median.)

## Step 6 — Isolate the Total Assets Column

In [ ]:
# assets vector
assets <- data %>%
  pull(total_assets)

head(assets)
summary(assets)

## Step 7 — Find the Median Assets

In [ ]:
# median assets (two equivalent ways)
median_assets <- median(assets)
# median_assets <- quantile(assets, 0.5)

median_assets

## Step 8 — Split into Small-Bank and Large-Bank NPL Vectors

In [ ]:
# small bank npl rate vector
small_npl <- data %>%
  filter(total_assets <= median_assets) %>%
  pull(npl_rate)

# large bank npl rate vector
large_npl <- data %>%
  filter(total_assets > median_assets) %>%
  pull(npl_rate)

length(small_npl)
length(large_npl)

## Step 9 — Quartiles of the Small-Bank Group

In [ ]:
# small bank npl quartiles
small_npl_quartiles <- quantile(small_npl)
small_npl_quartiles

## Step 10 — Quartiles of the Large-Bank Group

In [ ]:
# large bank npl quartiles
large_npl_quartiles <- quantile(large_npl)
large_npl_quartiles

## Step 11 — Histograms of the Two Groups

In [ ]:
# plot small bank histogram
hist(small_npl,
     col   = "red",
     main  = "NPL Rate — Small Banks (≤ median assets)",
     xlab  = "NPL Rate (%)",
     xlim  = range(c(small_npl, large_npl)))

# plot large bank histogram
hist(large_npl,
     col   = "blue",
     main  = "NPL Rate — Large Banks (> median assets)",
     xlab  = "NPL Rate (%)",
     xlim  = range(c(small_npl, large_npl)))

## Step 12 — Interpret 9 % in Each Group

Compare 9 % against `small_npl_quartiles` and `large_npl_quartiles`.

Typical pattern (observed with this data):

- In the **large-bank** group the whole distribution is shifted left (lower NPL); 9 % may fall in the upper half of that group.
- In the **small-bank** group 9 % is often near the centre or lower half.

This is visual evidence that larger banks tend to report lower NPL rates on average (association, not proven causation).

## Alternate Code Paths

In [ ]:
# ALTERNATE 1 — base-R extraction + fivenum / summary
npl_base <- data$npl_rate          # or data[["npl_rate"]]
fivenum(npl_base)                  # Tukey five-number summary
summary(npl_base)

# ALTERNATE 2 — logical indexing instead of filter()
small_npl_alt  <- npl_rate[assets <= median_assets]
large_npl_alt  <- npl_rate[assets >  median_assets]
quantile(small_npl_alt)
quantile(large_npl_alt)

# ALTERNATE 3 — create a factor and use tapply / split
size_group <- factor(ifelse(assets <= median_assets, "Small", "Large"),
                     levels = c("Small", "Large"))
tapply(npl_rate, size_group, quantile)
# or
split_npl <- split(npl_rate, size_group)
lapply(split_npl, quantile)

# ALTERNATE 4 — ggplot2 histograms (faceted)
library(ggplot2)
plot_df <- data.frame(
  npl   = c(small_npl, large_npl),
  group = rep(c("Small Banks", "Large Banks"),
              c(length(small_npl), length(large_npl)))
)
ggplot(plot_df, aes(x = npl, fill = group)) +
  geom_histogram(bins = 15, colour = "white", alpha = 0.85) +
  facet_wrap(~ group, ncol = 1) +
  scale_fill_manual(values = c("Small Banks" = "#e74c3c", "Large Banks" = "#3498db")) +
  labs(title = "NPL Rate by Bank Size Group",
       x = "NPL Rate (%)", y = "Count") +
  theme_minimal() +
  theme(legend.position = "none")

## More Practice — Solutions

In [ ]:
# 1. IQR of overall and of each group
iqr_overall <- IQR(npl_rate)
iqr_small   <- IQR(small_npl)
iqr_large   <- IQR(large_npl)
cat("IQR overall:", iqr_overall, "\n")
cat("IQR small banks:", iqr_small, "\n")
cat("IQR large banks:", iqr_large, "\n")

# 2. Mean vs median (skewness indicator)
cat("\nSmall banks — mean:", mean(small_npl), " median:", median(small_npl), "\n")
cat("Large banks — mean:", mean(large_npl), " median:", median(large_npl), "\n")

# 3. Mid-sized banks (25th–75th percentile of assets)
q25_assets <- quantile(assets, 0.25)
q75_assets <- quantile(assets, 0.75)
mid_npl <- data %>%
  filter(total_assets > q25_assets, total_assets <= q75_assets) %>%
  pull(npl_rate)
cat("\nMid-size group size:", length(mid_npl), "\n")
quantile(mid_npl)

# 4. Spearman correlation (association direction and strength)
cor(npl_rate, assets, method = "spearman")

## Simulation Section — Parameterised Experiment

Change `split_prob`, `noise_sd` or `n_sim` and re-run the cell to see how the gap between small- and large-bank NPL rates changes.

In [ ]:
# SIMULATION parameters (edit these)
set.seed(42)
split_prob <- 0.5      # assets quantile used as threshold (0.5 = median)
noise_sd   <- 0.8      # SD of Gaussian noise added to NPL rate
n_sim      <- 300      # number of Monte-Carlo replicates

# Observed difference under the chosen split (with optional noise)
npl_noisy  <- npl_rate + rnorm(length(npl_rate), 0, noise_sd)
threshold  <- quantile(assets, split_prob)
obs_diff   <- median(npl_noisy[assets <= threshold]) -
              median(npl_noisy[assets >  threshold])   # Small − Large
cat("Observed median difference (Small - Large) under current params:",
    round(obs_diff, 2), "percentage points\n")

# Monte-Carlo distribution of the difference
diffs <- replicate(n_sim, {
  npl_n <- npl_rate + rnorm(length(npl_rate), 0, noise_sd)
  thr   <- quantile(assets, split_prob)
  median(npl_n[assets <= thr]) - median(npl_n[assets > thr])
})

hist(diffs,
     main = paste0("Monte-Carlo: Small-Large median NPL difference\n",
                   "(split_prob = ", split_prob, ", noise_sd = ", noise_sd, ")"),
     xlab = "Median NPL (Small banks) − Median NPL (Large banks)",
     col  = "purple", border = "white")
abline(v = mean(diffs), col = "red", lwd = 2)
abline(v = quantile(diffs, c(0.025, 0.975)), col = "orange", lty = 2)
legend("topleft",
       legend = c(paste("Mean diff =", round(mean(diffs), 2)),
                  "95% Monte-Carlo interval"),
       col = c("red", "orange"), lty = c(1, 2), lwd = c(2, 1), bty = "n")

cat("Mean simulated difference:", round(mean(diffs), 2), "\n")
cat("95% MC interval: [", round(quantile(diffs, 0.025), 2), ",",
    round(quantile(diffs, 0.975), 2), "]\n")

## Audience-Adapted Takeaways

**For a Credit Risk analyst / technical colleague**  
NPL rates are moderately right-skewed overall. Splitting at the median of total assets produces two shifted distributions: small banks show a higher median NPL and greater dispersion, while large banks cluster at lower NPL levels. Spearman correlation between assets and NPL is negative. Next steps could include a simple linear or log-linear model, controls for business model / geography, and formal tests of stochastic dominance. From a risk-appetite perspective the dispersion among smaller banks warrants closer monitoring of the upper tail.

**For a busy executive or Board member**  
**Headline:** Banks above the median asset size report lower non-performing loan rates on average than smaller banks.  
The side-by-side histograms make the gap visible even without statistics training.  
**Implication:** Scale and diversification appear associated with better average asset quality; portfolio strategy and capital planning should continue to recognise that smaller banks can exhibit both higher average NPL and greater variability.